In [129]:
import pandas as pd
import numpy as np
df = pd.read_csv("C:/Users/NewAdminUser/Downloads/transaction_data.csv")

In [130]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36086 entries, 0 to 36085
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   customer_id           36086 non-null  int64  
 1   transaction_count     36086 non-null  int64  
 2   promo_usage_count     36086 non-null  int64  
 3   quantity              36086 non-null  int64  
 4   avg_order_value       36086 non-null  float64
 5   first_purchase_date   36086 non-null  object 
 6   last_purchase_date    36086 non-null  object 
 7   purchase_recency      36086 non-null  float64
 8   customer_tenure       36086 non-null  float64
 9   payment_method_count  36086 non-null  int64  
 10  total_spent           36086 non-null  float64
dtypes: float64(4), int64(5), object(2)
memory usage: 3.0+ MB


In [131]:
df["total_spent"].sum()

25577540.15

__We made over $2.56 million in 2021__

<h3 style="color:yellow">1. Divide customers into tiers based on how much they spent within the year, we can't treat everyone the same</h3>


In [132]:
high_cutoff = df['total_spent'].quantile(0.80)
mid_cutoff = df['total_spent'].quantile(0.50)




print(f"High Value Threshold: ${high_cutoff:.2f}")
print(f"Mid Value Threshold: ${mid_cutoff:.2f}")

def assign_tier(value, high, mid):
    if value >= high:
        return 'High'
    elif value >= mid:
        return 'Mid'
    else:
        return 'Low'

df['customer_tier'] = df['total_spent'].apply(lambda x: assign_tier(x, high_cutoff, mid_cutoff))

revenue_by_tier = (
    df.groupby('customer_tier')['total_spent']
      .agg(['sum', 'mean', 'count'])
      .sort_values('sum', ascending=False)
)

# Revenue percentage contribution
revenue_by_tier['revenue_%'] = (
    revenue_by_tier['sum'] / revenue_by_tier['sum'].sum()
) * 100

print(revenue_by_tier.round(2))

df.sample(5)

High Value Threshold: $819.40
Mid Value Threshold: $130.65
                       sum     mean  count  revenue_%
customer_tier                                        
High           20860258.95  2890.03   7218      81.56
Mid             3915686.95   361.73  10825      15.31
Low              801594.26    44.43  18043       3.13


,customer_id,transaction_count,promo_usage_count,quantity,avg_order_value,first_purchase_date,last_purchase_date,purchase_recency,customer_tenure,payment_method_count,total_spent,customer_tier
32074,64382,45,38,132,96.441,1/8/2021 20:39,12/26/2021 21:41,5.096174,357.138952,3,8872.598,High
26284,31347,6,6,7,14.316,3/22/2021 20:21,12/18/2021 1:46,13.925985,284.151610,1,100.213,Low
17969,54542,3,1,13,84.953,3/15/2021 8:24,11/24/2021 12:18,37.487071,291.649849,2,509.717,Mid
3700,14129,1,0,3,21.881,6/24/2021 7:19,6/24/2021 7:19,190.694858,190.694858,1,43.762,Low
16210,15169,4,7,8,36.208,3/9/2021 20:28,11/16/2021 22:11,45.075199,297.147178,4,289.668,Mid


In [133]:


# Sort descending by spend
df_sorted = df.sort_values('total_spent', ascending=False).reset_index(drop=True)

# Cumulative revenue
df_sorted['cum_revenue'] = df_sorted['total_spent'].cumsum()
df_sorted['cum_revenue_pct'] = (
    df_sorted['cum_revenue'] / df_sorted['total_spent'].sum()
) * 100

# Cumulative customer percentage
df_sorted['cum_customer_pct'] = (
    (df_sorted.index + 1) / len(df_sorted)
) * 100

for p in [5, 10, 20, 30, 50]:
    revenue_share = df_sorted.loc[
        df_sorted['cum_customer_pct'] <= p, 
        'total_spent'
    ].sum() / df['total_spent'].sum() * 100
    
    print(f"Top {p}% customers generate {revenue_share:.2f}% of revenue")

Top 5% customers generate 46.30% of revenue
Top 10% customers generate 63.93% of revenue
Top 20% customers generate 81.55% of revenue
Top 30% customers generate 89.90% of revenue
Top 50% customers generate 96.87% of revenue


### How recent is their last purchase by the end of the year

In [134]:
recency_by_tier = (
    df.groupby('customer_tier')['purchase_recency']
      .agg(['mean', 'median', 'min', 'max'])
      .sort_index()
)

print(recency_by_tier.round(2))

                 mean  median   min     max
customer_tier                              
High            21.86   10.42  0.00  362.34
Low            110.34   85.29  0.01  364.95
Mid             39.83   22.77  0.00  363.26


### How long since they made first purchase (Recency)

In [135]:
tenure_by_tier = (
    df.groupby('customer_tier')['customer_tenure']
      .agg(['mean', 'median'])
      .sort_index()
)

print(tenure_by_tier.round(2))

                 mean  median
customer_tier                
High           316.06  349.38
Low            222.89  240.86
Mid            292.37  329.42


### Likelihood that customers will buy again

In [136]:
repeat_rate = (
    df.assign(is_repeat=df['transaction_count'] > 1)
      .groupby('customer_tier')['is_repeat']
      .mean() * 100
)

print(repeat_rate.round(2))

customer_tier
High    97.31
Low     53.24
Mid     93.09
Name: is_repeat, dtype: float64


### How many days before next purchase is made (Frequency)

In [137]:
# Converting dates to datetime
df['first_purchase_date'] = pd.to_datetime(df['first_purchase_date'])
df['last_purchase_date'] = pd.to_datetime(df['last_purchase_date'])

# Days between first and last purchase
df['days_active'] = (df['last_purchase_date'] - df['first_purchase_date']).dt.days

# Average days between transactions
# If only 1 transaction, it's set as NaN or 0
df['avg_days_between_txn'] = df.apply(
    lambda row: row['days_active'] / (row['transaction_count'] - 1)
    if row['transaction_count'] > 1 else np.nan,
    axis=1
)


avg_days_by_tier = df.groupby('customer_tier')['avg_days_between_txn'] \
                     .agg(['mean', 'median', 'min', 'max']) \
                     .sort_index()

print(avg_days_by_tier.round(2))

                 mean  median   min    max
customer_tier                             
High            37.03    22.0  2.02  289.0
Low            132.99   122.0  6.00  339.0
Mid             66.13    49.0  3.20  288.0


<hr/>

## Major strategies to increase retention and customer value

Now we have information about our main customer segments, We also want to be able to retain them as much as possible, reducing churn to a minimum

<h3 style="color:yellow">1. Flag mid-tier customers, that are close to hitting the high-value spend threshold</h3>
We can target them specifically with special offers, discounts, product recommendations or incentives to make them spend more, increasing the number of regular high-tier customers rather than using marketing spend on all customers at once; It can be more costly and less focused

In [138]:
df['spend_gap_to_high'] = high_cutoff - df['total_spent']
upgrade_candidates = df[
    (df['customer_tier'] == 'Mid') &
    (df['spend_gap_to_high'] > 0) &
    (df['spend_gap_to_high'] < 100)  
]


upgrade_candidates["total_spent"].describe()

count    632.000000
mean     768.078438
std       29.452294
min      719.581000
25%      741.604500
50%      767.145000
75%      794.197000
max      819.334000
Name: total_spent, dtype: float64

The above output tells us that these 632 customers need to spend at least $100 within a time period (maybe a year) to be classified as high-tier. That's an extra $63,200 boost to revenue in that period

<hr/>

<h3 style="color:yellow">2. Trigger Re-ordering nudge to customers who are at-risk of churning</h3>

Based on past transactions, each customer has an average number of days between which they make an order. Depending on how much they exceed their average, we may have to resurface the idea of a purchase to these customers, maybe through reminders, retarget ads, etc.


In [ ]:
def churn_risk(row):
    if row['purchase_recency'] > 1.5 * row['avg_days_between_txn']:
        return "High Risk"
    elif row['purchase_recency'] > 1.2 * row['avg_days_between_txn']:
        return "Medium Risk"
    else:
        return "Healthy"

df.apply(churn_risk, axis=1).value_counts()

Healthy        35421
High Risk        361
Medium Risk      304
Name: count, dtype: int64

if a customer average day between purchase is 20 days, they are at high churn risk if they haven't purchased in 30 days

Main things to monitor:

+ Avg days between purchase
+ % revenue from repeat buyers
+ Revenue from top customers
+ Churn-risk count